# Lab 3: Churn to policy-grounded retention offers

Track: A+B. This clean run loads the registered model, predicts five real synthetic-course profiles from the verified Athena dataset, retrieves policy, and invokes Bedrock (or reuses an identical recorded generation). Historical experiment scores are read from the actual harness CSV; this notebook does not fabricate or re-run those scores.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display, Markdown
repo = Path.cwd()
if not (repo / 'rag_pipeline.py').exists():
    repo = repo.parent
sys.path.insert(0, str(repo))
from rag_index import sha256, verify_artifacts, write_json
from rag_retrieval import PolicyRetriever, query_spec
from rag_pipeline import OfferPipeline
evidence = repo / 'docs/lab3-evidence'
print('Repository:', repo.name)
print('XGBoost:', xgb.__version__)

Repository: cs401r-easonzhu
XGBoost: 3.2.0


## 1. The same evaluated churn model

The input profiles come from the successful Athena query named below. Categories were derived from valid transaction dates on or before 2026-04-01, because top_categories is not in the Feature Group. The model hash must match the registered training lineage.

In [2]:
lineage = json.loads((evidence / 'training-lineage.json').read_text(encoding='utf-8'))
model_path = repo / 'artifacts/lab3-churn/model.xgb'
assert sha256(model_path.read_bytes()) == lineage['model_sha256']
model = xgb.Booster()
model.load_model(bytearray(model_path.read_bytes()))
inputs = json.loads((evidence / 'real-customer-profiles.json').read_text(encoding='utf-8'))
profiles = inputs['profiles']
X = pd.DataFrame(profiles)[lineage['feature_columns']]
predicted = model.predict(xgb.DMatrix(X, feature_names=lineage['feature_columns']))
np.testing.assert_allclose(predicted, [p['churn_probability'] for p in profiles], atol=1e-7)
print('Athena query:', lineage['athena_query_execution_id'])
print('Registered model SHA256:', lineage['model_sha256'])
display(pd.DataFrame(profiles)[['customer_id','loyalty_tier','churn_probability','top_categories']])

Athena query: a4a936c4-dcbe-46d7-bf7c-d2d1627e6c9f
Registered model SHA256: 8ef39c2469667a6e718318826294962f88679b4ac30c3c259ea1d6b88a1d95dc


,customer_id,loyalty_tier,churn_probability,top_categories
0,CUST-10000299,Silver,0.828145,"[Beauty, Home, Electronics]"
1,CUST-10000324,Bronze,0.828145,"[Apparel, Grocery, Home]"
2,CUST-10007187,Gold,0.828145,"[Home, Footwear, Beauty]"
3,CUST-10002887,Platinum,0.827040,"[Home, Beauty, Toys]"
4,CUST-10000350,Silver,0.828145,[Apparel]


## 2. Verify and retrieve policy

The final retrieval configuration combines the top three C results with the top three B results, deduplicated by chunk ID. C retrieves entitlements and shipping limits; B adds promotion and final-sale restrictions. This increases the context budget from four to at most six chunks, so the ablation does not isolate query composition from budget.

In [3]:
manifest = verify_artifacts(repo / 'northstar-policy-docs', repo / 'artifacts/lab3-rag')
retriever = PolicyRetriever()
print('Policy matrix:', manifest['matrix_shape'])
print('Persisted FAISS vectors:', retriever.index.ntotal)
for strategy in ['A','B','C']:
    result = retriever.retrieve(profiles[0], strategy)
    print(strategy, [(h['doc_id'], h['section_path'], round(h['score'],4)) for h in result['results']])

Policy matrix: [48, 1024]
Persisted FAISS vectors: 48
A [('POL-LOY-011', '4. Tier Benefits', 0.6835), ('KB-FAQ-022', 'Loyalty', 0.5128), ('POL-LOY-011', '2. Tiers', 0.4498), ('POL-RET-004', '1. Standard Return Window / 1.1 Extended Window for Platinum Members', 0.4149)]
B [('POL-LOY-011', '5. Promotional Offers / 5.1 Retention Offers', 0.5365), ('POL-LOY-011', '5. Promotional Offers', 0.5235), ('POL-RET-004', '4. Non-Returnable Merchandise', 0.4606), ('KB-FAQ-022', 'Returns', 0.3238)]
C [('POL-SHP-007', '2. Free Shipping Thresholds', 0.4752), ('POL-LOY-011', '4. Tier Benefits', 0.4743), ('POL-LOY-011', '2. Tiers', 0.4271), ('POL-LOY-011', '2. Tiers / 2.1 Tier Upgrades', 0.369)]


## 3. Generate five traceable offers

Claude Haiku 4.5 proposes copy through Converse. A bounded output check uses the retrieved points table and observed category. Unsafe proposals fall back to a policy-rendered two-sentence offer; delivery_mode discloses this. Raw model output, tokens and request IDs remain in the saved records. These are offer drafts, not issued coupons or account changes.

In [4]:
pipeline = OfferPipeline(strategy='CB', prompt_variant='controlled')
offers = [pipeline(p, 'Generate a concise retention offer for this model-flagged customer.') for p in profiles]
write_json(evidence / 'offers-real.json', offers)
for result in offers:
    assert result['body'].startswith('If your current membership is ')
    assert result['customer_profile']['top_categories'][0].lower() in result['body'].lower()
    display(Markdown('**' + result['customer_profile']['customer_id'] + '** — ' + result['delivery_mode'] + '\n\n' + result['answer']))
    print('Policy sources:', [h['chunk_id'] for h in result['retrieved_chunks']])
print('Offers:', len(offers), 'new generation calls:', sum(not r['generation_cache_hit'] for r in offers))

**CUST-10000299** — policy_template_fallback

Subject: Explore Beauty with Summit Rewards
Body: If your current membership is Silver, you earn 1.25× points on qualifying Beauty purchases. Explore Beauty at NorthStar and check your current membership eligibility before using these benefits.

Policy sources: ['POL-SHP-007:003:c163e7eb41', 'POL-LOY-011:010:cdbdbe1ea2', 'POL-LOY-011:003:8c4328c83f', 'POL-LOY-011:013:c2d7a201cd', 'POL-LOY-011:012:d96352d7c0', 'POL-RET-004:007:49b78de484']


**CUST-10000324** — policy_template_fallback

Subject: Explore Apparel with Summit Rewards
Body: If your current membership is Bronze, you earn 1× points on qualifying Apparel purchases. Explore Apparel at NorthStar and check your current membership eligibility before using these benefits.

Policy sources: ['POL-LOY-011:010:cdbdbe1ea2', 'POL-SHP-007:003:c163e7eb41', 'POL-LOY-011:003:8c4328c83f', 'POL-LOY-011:013:c2d7a201cd', 'POL-LOY-011:012:d96352d7c0', 'POL-RET-004:007:49b78de484']


**CUST-10007187** — policy_template_fallback

Subject: Explore Home with Summit Rewards
Body: If your current membership is Gold, you earn 1.5× points on qualifying Home purchases. Explore Home at NorthStar and check your current membership eligibility before using these benefits.

Policy sources: ['POL-LOY-011:004:773059e1b0', 'POL-LOY-011:010:cdbdbe1ea2', 'POL-SHP-007:003:c163e7eb41', 'POL-LOY-011:013:c2d7a201cd', 'POL-LOY-011:012:d96352d7c0', 'POL-RET-004:007:49b78de484']


**CUST-10002887** — policy_template_fallback

Subject: Explore Home with Summit Rewards
Body: If your current membership is Platinum, you earn 2× points on qualifying Home purchases. Explore Home at NorthStar and check your current membership eligibility before using these benefits.

Policy sources: ['POL-SHP-007:003:c163e7eb41', 'POL-LOY-011:010:cdbdbe1ea2', 'POL-LOY-011:003:8c4328c83f', 'POL-LOY-011:013:c2d7a201cd', 'POL-LOY-011:012:d96352d7c0', 'POL-RET-004:007:49b78de484']


**CUST-10000350** — policy_template_fallback

Subject: Explore Apparel with Summit Rewards
Body: If your current membership is Silver, you earn 1.25× points on qualifying Apparel purchases. Explore Apparel at NorthStar and check your current membership eligibility before using these benefits.

Policy sources: ['POL-SHP-007:003:c163e7eb41', 'POL-LOY-011:010:cdbdbe1ea2', 'POL-LOY-011:003:8c4328c83f', 'POL-LOY-011:013:c2d7a201cd', 'POL-LOY-011:012:d96352d7c0', 'POL-RET-004:007:49b78de484']
Offers: 5 new generation calls: 0


## 4. Actual repeated harness results

The judge and generator are the Haiku 4.5 US inference profile. Each configuration uses all four original cases twice; cached generation is held fixed while judge calls are repeated. The teacher removed the historical fixed score gates on October 12. Min/max below span cases and repetitions; per-run means are also shown.

In [5]:
scores = pd.read_csv(evidence / 'rag_eval.csv')
metrics = ['faithfulness','answer_relevancy','context_recall']
assert scores[metrics].notna().all().all()
assert scores.groupby(['strategy','repeat']).size().eq(4).all()
print('Judge:', scores.judge_model.iloc[0])
display(scores.groupby('strategy')[metrics].agg(['mean','min','max']))
display(scores.groupby(['strategy','repeat'])[metrics].mean())

Judge: us.anthropic.claude-haiku-4-5-20251001-v1:0


faithfulness                     answer_relevancy            \
                      mean       min       max             mean       min   
strategy                                                                    
A                 0.183985  0.100000  0.307692         0.347177  0.291944   
B                 0.179167  0.000000  0.500000         0.394921  0.337665   
C                 0.308838  0.090909  0.545455         0.359561  0.296740   
CB                0.297475  0.090909  0.454545         0.360169  0.263346   
CB_controlled     0.031250  0.000000  0.125000         0.308122  0.218964   
CB_hardened       0.137500  0.000000  0.300000         0.369371  0.257216   

                        context_recall                 
                    max           mean       min  max  
strategy                                               
A              0.425992       0.483333  0.200000  1.0  
B              0.479380       0.387500  0.000000  0.8  
C              0.479029       0.483333  0.200000  1.0  
CB             0.532558       0.683333  0.333333  1.0  
CB_controlled  0.428025       0.683333  0.333333  1.0  
CB_hardened    0.476730       0.683333  0.333333  1.0

faithfulness  answer_relevancy  context_recall
strategy      repeat                                                
A             1           0.178628          0.341227        0.483333
              2           0.189342          0.353126        0.483333
B             1           0.229167          0.394985        0.387500
              2           0.129167          0.394857        0.387500
C             1           0.320202          0.358001        0.483333
              2           0.297475          0.361121        0.483333
CB            1           0.297475          0.360290        0.683333
              2           0.297475          0.360048        0.683333
CB_controlled 1           0.031250          0.297541        0.683333
              2           0.031250          0.318702        0.683333
CB_hardened   1           0.106250          0.368932        0.683333
              2           0.168750          0.369811        0.683333

## 5. Interpretation and limits

A and C retrieve benefits but miss final-sale clauses; B retrieves constraints but misses the tier table. Adding the B pass improves context recall in the measured C-to-CB comparison, while the raw offers remain unsafe. The points-only controlled delivery is deliberately less expressive. Its fallback is application code, not an undisclosed LLM success. RAGAS relevance compares marketing instructions to reverse-generated questions; context-recall references contain business commentary absent from policy; faithfulness is sensitive to marketing filler and how conditional claims are interpreted. No statement here implies customer response lift or production approval.

In [6]:
registry_export = evidence / 'model-registry.json'
print('Training metrics:', json.loads((evidence / 'evaluation_metrics.json').read_text(encoding='utf-8'))['auc_roc'])
print('Evidence outputs are saved under docs/lab3-evidence/')
print('Clean run complete')

Training metrics: 0.7838
Evidence outputs are saved under docs/lab3-evidence/
Clean run complete
